In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/GoogleNQ_UND_Gemini_Ragas.csv')
df.to_json("./GoogleNQ_gpt4o_rewriting_files/GoogleNQ_UND_Gemini_Ragas.jsonl", orient="records", lines=True)

## Rewriting with GPT-4o
GPT-4o rewriting, then Gemini QA later

In [3]:
from helper_functions_qr import modification_in_batch

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = "gpt-4o-2024-11-20"
input_file = "./GoogleNQ_gpt4o_rewriting_files/GoogleNQ_UND_Gemini_Ragas.jsonl"
output_file = "./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl"

In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'short_answers', client, model)

Total samples to process: 458
Batch size: 3


Processing batches:   0%|          | 0/153 [00:00<?, ?it/s]

Processing batches: 100%|██████████| 153/153 [17:27<00:00,  6.85s/it]


All batch processing completed! Total processed: 458 samples
Results saved to: ./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl


In [7]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,where does the modern view of history originat...,Where does the concept of the modern view of h...,['approximately in the early 16th century'],['* The Enlightenment\n* 19th-century Germ...,The query asks about the 'modern view of histo...,0.000000,0,0.00
1,when did the first wireless beats come out,When did the first wireless Beats by Dre headp...,['October 2012'],['* 2014'],The query is ambiguous due to several factors:...,0.000000,0,0.00
2,this inventor co-created the film fred ott’s s...,"Which inventor, known for his contributions to...",['Edison'],['* William K.L. Dickson'],The query refers to an 'inventor' who co-creat...,0.000000,0,0.00
3,what is the collection of the districts to the...,What are the regions or territories located to...,['Golan Heights' 'Jordan'],['* Transjordan'],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50
4,factories that assemble parts made in other co...,What are special economic zones where factorie...,['special economic zones'],['* Assembly plant\n* Contract manufacture...,The query lacks specificity regarding critical...,0.000000,0,0.25
...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,Where does the British royal family get their ...,['the hereditary revenues of the Crown'],['* Government grants or taxpayer funds\n* ...,The query is underspecified because 'royal fam...,0.095238,0,0.75
454,where does the show the path take place,Where does the TV show 'The Path' (2016-2018) ...,['Upstate New York'],['* Upstate New York'],The query asks for the location of the show 'T...,1.000000,1,1.00
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses a...,['Frank Ferrer'],['* Frank Ferrer'],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00
456,what is the meaning of auv in cars,What does the acronym 'AUV' stand for in the c...,['action utility vehicles'],['* Asian Utility Vehicle'],The query asks for the meaning of 'auv' in the...,0.333333,0,0.00


## Modified queries QA using Gemini-2.5-Flash

### Loading modified data

In [4]:
modified_set = load_dataset("json",
    data_files="./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


modified_set = modified_set.remove_columns(["model_new_answer"])

modified_set.to_json(
    "./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl",
    orient="records",
    lines=True
)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 112.33ba/s]


455911

### Implementation

In [5]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [7]:
ask_short_answer('who are you?', client, model="gemini-2.5-flash")

['*   I am a large language model, trained by Google.']

In [8]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gemini-2.5-flash",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 46/46 [17:51<00:00, 23.30s/it]


In [9]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 97.43ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,where does the modern view of history originat...,Where does the concept of the modern view of h...,['approximately in the early 16th century'],['* The Enlightenment\n* 19th-century Germ...,The query asks about the 'modern view of histo...,0.000000,0,0.00,[* The Renaissance]
1,when did the first wireless beats come out,When did the first wireless Beats by Dre headp...,['October 2012'],['* 2014'],The query is ambiguous due to several factors:...,0.000000,0,0.00,[December 2011]
2,this inventor co-created the film fred ott’s s...,"Which inventor, known for his contributions to...",['Edison'],['* William K.L. Dickson'],The query refers to an 'inventor' who co-creat...,0.000000,0,0.00,[- William K.L. Dickson]
3,what is the collection of the districts to the...,What are the regions or territories located to...,['Golan Heights' 'Jordan'],['* Transjordan'],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,[* Golan Heights\n* Transjordan]
4,factories that assemble parts made in other co...,What are special economic zones where factorie...,['special economic zones'],['* Assembly plant\n* Contract manufacture...,The query lacks specificity regarding critical...,0.000000,0,0.25,[* Export Processing Zones\n* Free Trade Z...
...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,Where does the British royal family get their ...,['the hereditary revenues of the Crown'],['* Government grants or taxpayer funds\n* ...,The query is underspecified because 'royal fam...,0.095238,0,0.75,[* Sovereign Grant\n* Duchy of Lancaster (...
454,where does the show the path take place,Where does the TV show 'The Path' (2016-2018) ...,['Upstate New York'],['* Upstate New York'],The query asks for the location of the show 'T...,1.000000,1,1.00,[- Upstate New York]
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses a...,['Frank Ferrer'],['* Frank Ferrer'],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,[Frank Ferrer]
456,what is the meaning of auv in cars,What does the acronym 'AUV' stand for in the c...,['action utility vehicles'],['* Asian Utility Vehicle'],The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,[Action Utility Vehicle]


## Evaluations

### Squad EM+F1

In [11]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./GoogleNQ_gpt4o_rewriting_files/MODIFIED_GoogleNQ_UND_Gemini_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 458 examples [00:00, 60469.38 examples/s]


In [12]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./GoogleNQ_gpt4o_rewriting_files/MODIFIED_gpt4o_GoogleNQ_UND_Gemini_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./GoogleNQ_gpt4o_rewriting_files/MODIFIED_gpt4o_GoogleNQ_UND_Gemini_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 117.28ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,where does the modern view of history originat...,Where does the concept of the modern view of h...,['approximately in the early 16th century'],['* The Enlightenment\n* 19th-century Germ...,The query asks about the 'modern view of histo...,0.000000,0,0.00,[* The Renaissance],0,0.000000
1,when did the first wireless beats come out,When did the first wireless Beats by Dre headp...,['October 2012'],['* 2014'],The query is ambiguous due to several factors:...,0.000000,0,0.00,[December 2011],0,0.000000
2,this inventor co-created the film fred ott’s s...,"Which inventor, known for his contributions to...",['Edison'],['* William K.L. Dickson'],The query refers to an 'inventor' who co-creat...,0.000000,0,0.00,[- William K.L. Dickson],0,0.000000
3,what is the collection of the districts to the...,What are the regions or territories located to...,['Golan Heights' 'Jordan'],['* Transjordan'],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,[* Golan Heights\n* Transjordan],0,0.666667
4,factories that assemble parts made in other co...,What are special economic zones where factorie...,['special economic zones'],['* Assembly plant\n* Contract manufacture...,The query lacks specificity regarding critical...,0.000000,0,0.25,[* Export Processing Zones\n* Free Trade Z...,0,0.222222
...,...,...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,Where does the British royal family get their ...,['the hereditary revenues of the Crown'],['* Government grants or taxpayer funds\n* ...,The query is underspecified because 'royal fam...,0.095238,0,0.75,[* Sovereign Grant\n* Duchy of Lancaster (...,0,0.105263
454,where does the show the path take place,Where does the TV show 'The Path' (2016-2018) ...,['Upstate New York'],['* Upstate New York'],The query asks for the location of the show 'T...,1.000000,1,1.00,[- Upstate New York],1,1.000000
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses a...,['Frank Ferrer'],['* Frank Ferrer'],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,[Frank Ferrer],1,1.000000
456,what is the meaning of auv in cars,What does the acronym 'AUV' stand for in the c...,['action utility vehicles'],['* Asian Utility Vehicle'],The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,[Action Utility Vehicle],0,0.666667


In [17]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 30.13
New answers after modification F1 Score (avg): 51.12
Original answers Exact Match (avg): 20.31
Original answers F1 Score (avg): 38.76
F1: t=4.745, p=0.0000
EM: t=3.442, p=0.0006


### Ragas AA

In [18]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [19]:
squad_scored_modified = load_dataset("json",
    data_files="./GoogleNQ_gpt4o_rewriting_files/MODIFIED_gpt4o_GoogleNQ_UND_Gemini_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_gpt4o_GoogleNQ_UND_Gemini_all_new_scores.csv")

Generating train split: 458 examples [00:00, 53960.43 examples/s]
Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 32.61ba/s]


424187

In [20]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 54.86
modified AA (avg): 64.47
AA: t=3.336, p=0.0009


## Re-Classification

In [3]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [4]:
reclassify_file = "./output_csv/MODIFIED_gpt4o_GoogleNQ_UND_Gemini_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.19s/it]


cuda


### Prepare prompts

In [6]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [7]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 458
Generation complete: 458 prompts
Average prompt length: 444 bytes (~111 tokens)

Analyze the following input user query:

{"query": "Where does the concept of the modern view of history, characterized by a focus on critical analysis of sources and the emergence of humanist perspectives, originate in terms of its historical development?"}

Please provide your analysis in the following JSON format:

{"query": "Where does the concept of the modern view of history, characterized by a focus on critical analysis of sources and the emergence of humanist perspectives, originate in terms of its historical development?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [9]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 92/92 [55:26<00:00, 36.16s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,where does the modern view of history originat...,Where does the concept of the modern view of h...,['approximately in the early 16th century'],['* The Enlightenment\n* 19th-century Germ...,The query asks about the 'modern view of histo...,0.000000,0,0.00,['* The Renaissance'],0.0,0.000000,0.25,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Where does the concept of the m...",fully specified
1,when did the first wireless beats come out,When did the first wireless Beats by Dre headp...,['October 2012'],['* 2014'],The query is ambiguous due to several factors:...,0.000000,0,0.00,['December 2011'],0.0,0.000000,0.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""When did the first wireless Bea...",fully specified
2,this inventor co-created the film fred ott’s s...,"Which inventor, known for his contributions to...",['Edison'],['* William K.L. Dickson'],The query refers to an 'inventor' who co-creat...,0.000000,0,0.00,['- William K.L. Dickson'],0.0,0.000000,0.25,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""Which inventor, known for his c...",fully specified
3,what is the collection of the districts to the...,What are the regions or territories located to...,['Golan Heights' 'Jordan'],['* Transjordan'],"The query refers to 'the Jordan River,' which ...",0.000000,0,0.50,['* Golan Heights\n* Transjordan'],0.0,0.666667,0.50,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the regions or territo...",fully specified
4,factories that assemble parts made in other co...,What are special economic zones where factorie...,['special economic zones'],['* Assembly plant\n* Contract manufacture...,The query lacks specificity regarding critical...,0.000000,0,0.25,['* Export Processing Zones\n* Free Trade ...,0.0,0.222222,0.25,"<think>\nOkay, let's tackle this query analysi...","{\n ""query"": ""What are special economic zones...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,where do royal families get their money from,Where does the British royal family get their ...,['the hereditary revenues of the Crown'],['* Government grants or taxpayer funds\n* ...,The query is underspecified because 'royal fam...,0.095238,0,0.75,['* Sovereign Grant\n* Duchy of Lancaster ...,0.0,0.105263,0.75,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Where does the British royal fa...",underspecified
454,where does the show the path take place,Where does the TV show 'The Path' (2016-2018) ...,['Upstate New York'],['* Upstate New York'],The query asks for the location of the show 'T...,1.000000,1,1.00,['- Upstate New York'],1.0,1.000000,1.00,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""Where does the TV show 'The Pat...",fully specified
455,who is the drummer for guns and roses,Who is the current drummer for Guns N' Roses a...,['Frank Ferrer'],['* Frank Ferrer'],The query asks for the 'drummer' for Guns N' R...,1.000000,1,1.00,['Frank Ferrer'],1.0,1.000000,1.00,"<think>\nOkay, let's see. The user is asking w...","{\n ""query"": ""Who is the current drummer for ...",fully specified
456,what is the meaning of auv in cars,What does the acronym 'AUV' stand for in the c...,['action utility vehicles'],['* Asian Utility Vehicle'],The query asks for the meaning of 'auv' in the...,0.333333,0,0.00,['Action Utility Vehicle'],0.0,0.666667,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What does the acronym 'AUV' sta...",fully specified


In [10]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.855895
underspecified     0.144105
Name: proportion, dtype: float64

In [11]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    392
underspecified      66
Name: count, dtype: int64

In [12]:
test_df.to_csv('./output_csv/GoogleNQ_UND_gpt4o_rewritten_reclassified.csv')